# Posições e deslocamentos dos motoristas (Silver)

Explode os snapshots de `tb_ofertas` em uma linha por motorista e snapshot, com estado operacional, posição GPS e flags de qualidade. Em seguida calcula o deslocamento entre posições consecutivas do mesmo motorista, base para medir quilômetros sem passageiro.

## Fluxo dos dados

* Origem: `cat_mobilidade_urbana_bronze.db_frota.tb_ofertas`
* Destinos: `cat_mobilidade_urbana_silver.db_frota.tb_posicoes_motoristas` e `cat_mobilidade_urbana_silver.db_frota.tb_deslocamentos_motoristas`

## Evidências que sustentam os tratamentos (perfil de 14/10/2026)

* **Cadência:** 262.708 snapshots em 365 dias locais; intervalo mediano de 120 s, p99 de 123 s, só 5 lacunas acima de 10 min.
* **Identidade do ofertante:** `ofertaList.codigo` coincide com 431 dos 450 códigos e com todos os `cD_MOTORISTA` das solicitações, e com apenas 1 `cD_VEICULO`. É tratado como **motorista**. Em 89% dos dias o motorista usa um único veículo; um veículo é dividido por mais de um motorista em 22% dos dias. A unidade móvel rastreada é o motorista com o veículo que está usando.
* **Estado:** comparando cada snapshot com `tb_sumarios` (mesmo `codigo`), as contagens batem em 99,999% dos snapshots: `status=1` = `qtdOnline`; `status=3` sem passageiro = `qtdACaminho`; `status=3` com passageiro = `qtdComPassageiro`; `status=4` = `qtdSelecionado`; `status=5` = `qtdIndisponivel`. `qtdOffline` é sempre zero: motorista offline não aparece na lista. O enum é inferido empiricamente, não documentado pela empresa.
* **Posição:** idade mediana do GPS de 3 s (p99 de 124 s); nenhuma coordenada nula.

Os deslocamentos somam distâncias Haversine entre pontos amostrados a cada ~2 min. É uma **estimativa por GPS amostrado**, tende a subestimar a rodagem real e não substitui odômetro (RN012).

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

## 1. Explosão dos snapshots

In [0]:
snapshots = spark.table(f"{BRONZE}.tb_ofertas").select(
    F.col("codigo").alias("id_snapshot"),
    F.col("codigoRegiao").cast("int").alias("id_regiao"),
    F.to_timestamp("dataHora").alias("ts_snapshot"),
    F.explode("ofertaList").alias("o"),
)

posicoes_brutas = snapshots.select(
    "id_snapshot",
    "ts_snapshot",
    "id_regiao",
    F.col("o.codigo").cast("bigint").alias("id_motorista"),
    F.col("o.status").cast("int").alias("cod_status"),
    F.col("o.passageiroEmbarcado").alias("fl_embarcado"),
    F.col("o.codigoSolicitacao").cast("bigint").alias("id_solicitacao"),
    F.col("o.codigoProximaSolicitacao").cast("bigint").alias("id_proxima_solicitacao"),
    F.col("o.latitude").cast("double").alias("latitude"),
    F.col("o.longitude").cast("double").alias("longitude"),
    F.col("o.h3Index").alias("h3_fonte"),
    F.to_timestamp("o.dataPosicaoGPS").alias("ts_posicao_gps"),
    F.to_timestamp("o.dataAtualizacao").alias("ts_atualizacao"),
    F.col("o.tempoSegundosViagemRestante").cast("int").alias("seg_viagem_restante"),
)

## 2. Estado, qualidade e zona

* `fl_coordenada_valida`: coordenada dentro do território brasileiro e diferente de (0, 0).
* `fl_posicao_atual`: idade da posição entre -60 s e 300 s em relação ao snapshot.
* `fl_bbox_grande_vitoria`: retângulo **aproximado** da Grande Vitória, apenas para recorte exploratório; a fronteira oficial do experimento continua pendente (RN002).
* `h3_r8` e `h3_r7`: células H3 calculadas sobre a coordenada. A resolução final será escolhida pela análise de dispersão (RN030).

In [0]:
estado = (
    F.when(F.col("cod_status") == 1, "ONLINE")
    .when(F.col("cod_status") == 4, "SELECIONADO")
    .when((F.col("cod_status") == 3) & F.col("fl_embarcado"), "COM_PASSAGEIRO")
    .when(F.col("cod_status") == 3, "A_CAMINHO")
    .when(F.col("cod_status") == 5, "INDISPONIVEL")
    .otherwise("NAO_MAPEADO")
)
coordenada_valida = (
    F.col("latitude").between(-34.0, 6.0)
    & F.col("longitude").between(-74.0, -34.0)
    & ~((F.col("latitude") == 0) & (F.col("longitude") == 0))
)

ordem_dup = W.partitionBy("id_snapshot", "id_motorista").orderBy(F.col("ts_atualizacao").desc_nulls_last())
posicoes = (
    posicoes_brutas.where(F.col("id_motorista").isNotNull())
    .withColumn("_rn", F.row_number().over(ordem_dup))
    .where("_rn = 1")
    .drop("_rn")
    .withColumn("estado", estado)
    .withColumn("ts_posicao", F.coalesce("ts_posicao_gps", "ts_atualizacao", "ts_snapshot"))
    .withColumn("idade_posicao_seg", segundos_entre("ts_posicao", "ts_snapshot"))
    .withColumn("fl_coordenada_valida", F.coalesce(coordenada_valida, F.lit(False)))
    .withColumn("fl_posicao_atual", F.col("idade_posicao_seg").between(-60, 300))
    .withColumn(
        "fl_bbox_grande_vitoria",
        F.col("latitude").between(-20.85, -19.80) & F.col("longitude").between(-40.80, -40.00),
    )
    .withColumn("h3_r8", F.when(F.col("fl_coordenada_valida"), F.expr("h3_longlatash3(longitude, latitude, 8)")))
    .withColumn("h3_r7", F.expr("h3_toparent(h3_r8, 7)"))
    .withColumn("data_local", F.to_date("ts_snapshot"))
    .withColumn("hora_local", F.hour("ts_snapshot"))
    .withColumn("_ts_processamento", F.current_timestamp())
)

gravar(
    posicoes,
    f"{SILVER}.tb_posicoes_motoristas",
    "Uma linha por motorista e snapshot de ofertas (cadencia ~2 min), com estado inferido do enum status, "
    "posicao GPS, H3 e flags de qualidade. Motorista offline nao aparece nos snapshots.",
)

## 3. Deslocamentos entre posições consecutivas

Para cada motorista, ordena as posições válidas e atuais pelo snapshot e calcula o trecho até a posição seguinte.

* `categoria_km` = estado no **início** do trecho. Na troca de estado o trecho inteiro fica com o estado inicial; o erro máximo é um intervalo de snapshot (~2 min).
* Trecho válido: intervalo entre snapshots de até 300 s e velocidade implícita de até 150 km/h. Intervalos maiores são **lacunas** (km não mensurável, nunca zero, RN027).

In [0]:
validas = spark.table(f"{SILVER}.tb_posicoes_motoristas").where("fl_coordenada_valida AND fl_posicao_atual")
ordem = W.partitionBy("id_motorista").orderBy("ts_snapshot", "id_snapshot")

anterior = validas.select(
    "id_motorista",
    F.lag("id_snapshot").over(ordem).alias("id_snapshot_inicio"),
    F.col("id_snapshot").alias("id_snapshot_fim"),
    F.lag("ts_snapshot").over(ordem).alias("ts_snapshot_inicio"),
    F.col("ts_snapshot").alias("ts_snapshot_fim"),
    F.lag("ts_posicao").over(ordem).alias("ts_posicao_inicio"),
    F.col("ts_posicao").alias("ts_posicao_fim"),
    F.lag("latitude").over(ordem).alias("lat_inicio"),
    F.lag("longitude").over(ordem).alias("lon_inicio"),
    F.col("latitude").alias("lat_fim"),
    F.col("longitude").alias("lon_fim"),
    F.lag("estado").over(ordem).alias("estado_inicio"),
    F.col("estado").alias("estado_fim"),
    F.lag("id_solicitacao").over(ordem).alias("id_solicitacao_inicio"),
    F.col("id_solicitacao").alias("id_solicitacao_fim"),
    F.lag("h3_r8").over(ordem).alias("h3_r8_inicio"),
)

deslocamentos = (
    anterior.where(F.col("ts_snapshot_inicio").isNotNull())
    .withColumn("seg_intervalo", segundos_entre("ts_snapshot_inicio", "ts_snapshot_fim"))
    .withColumn("seg_posicao", segundos_entre("ts_posicao_inicio", "ts_posicao_fim"))
    .withColumn("km_haversine", haversine_km("lat_inicio", "lon_inicio", "lat_fim", "lon_fim"))
    .withColumn(
        "velocidade_kmh",
        F.col("km_haversine")
        / (F.when(F.col("seg_posicao") > 0, F.col("seg_posicao")).otherwise(F.col("seg_intervalo")) / 3600.0),
    )
    .withColumn(
        "motivo_invalido",
        F.when(F.col("seg_intervalo") > 300, "LACUNA_TEMPORAL")
        .when(F.col("seg_intervalo") <= 0, "TEMPO_NAO_POSITIVO")
        .when(F.col("velocidade_kmh") > 150, "VELOCIDADE_IMPLAUSIVEL"),
    )
    .withColumn("fl_passo_valido", F.col("motivo_invalido").isNull())
    .withColumn("categoria_km", F.col("estado_inicio"))
    .withColumn("data_local", F.to_date("ts_snapshot_inicio"))
    .withColumn("hora_local", F.hour("ts_snapshot_inicio"))
    .withColumn("metodo_distancia", F.lit("haversine_gps_amostrado_2min"))
    .withColumn("_ts_processamento", F.current_timestamp())
)

gravar(
    deslocamentos,
    f"{SILVER}.tb_deslocamentos_motoristas",
    "Trecho entre posicoes GPS consecutivas do mesmo motorista (Haversine sobre GPS amostrado ~2 min, "
    "estimativa que subestima rodagem). categoria_km = estado no inicio do trecho; lacunas acima de 300 s invalidas.",
)

## 4. Conferência agregada

In [0]:
resumo = (
    spark.table(f"{SILVER}.tb_deslocamentos_motoristas")
    .groupBy("categoria_km", "fl_passo_valido", "motivo_invalido")
    .agg(F.count("*").alias("trechos"), F.round(F.sum("km_haversine"), 1).alias("km"))
    .orderBy("categoria_km", "motivo_invalido")
)
for linha in resumo.collect():
    print(linha.asDict())